# Introducción a la Terminal para Data Analytics
**Duración sugerida:** 90–120 min  
**Nivel:** Básico → Intermedio  
**Requisitos:** Ninguno (se recomiendan nociones de archivos y carpetas)

> Este cuaderno es auto-contenido: combina teoría y práctica progresiva para que ganes confianza en la **línea de comandos** (CLI) aplicada al flujo de trabajo de analítica de datos.

## Objetivos de aprendizaje
Al finalizar, podrás:
- Explicar **qué es la terminal/CLI** y por qué es valiosa en análisis de datos.
- Navegar el sistema de archivos con comandos esenciales (**pwd, ls, cd, mkdir, rm, mv, cp**).
- Usar **redirecciones** y **pipes** para crear mini‑pipelines de datos (**>, >>, |**).
- Inspeccionar y transformar archivos de texto/CSV con utilidades (**head, tail, wc, cut, sort, uniq, grep**).
- Comprender **stdin/stdout/stderr**, **códigos de salida**, **comodines** y **expansión de rutas**.
- Administrar procesos básicos (**ps, kill**, ejecución en background) y revisar recursos (**top/htop**).
- Ejecutar tareas comunes de DA sin abrir un IDE: muestreos, conteos, filtros y “quick‑wins”.

## ¿Qué es la terminal y para qué se usa?
La **terminal** (o *línea de comandos / shell*) es una interfaz basada en texto para interactuar con el sistema operativo. Permite:
- **Automatizar** tareas repetitivas con scripts.
- **Encadenar** herramientas pequeñas para crear soluciones poderosas (*"pipes"*).
- Trabajar **rápido y con pocos recursos**, ideal para servidores, contenedores y *remote computing*.
- Integrarse con **git**, **Docker**, gestores de paquetes, y ejecutores de *jobs*.
- En analítica, acelerar **ingestas, exploraciones rápidas y pre‑procesos** antes de mover datos a notebooks.

**Shells comunes:** `bash`, `zsh` (Linux/macOS), `PowerShell`/`cmd` (Windows).  
Nota: En Windows puedes usar **PowerShell**, **Git Bash** o **WSL** para seguir este cuaderno.

**Temas detectados del notebook adjunto:**  
Tutorial de Comandos CLI en Jupyter Notebook, Introducción, 1. Comandos Básicos, cmd:ls, cmd:pwd, cmd:cd, cmd:mkdir, cmd:head, cmd:tail, cmd:find, cmd:grep, Ejercicio de Clase: Organización de Archivos, Ejercicio Adicional: Búsqueda y Filtrado, 2. Instalación y Configuración de Herramientas, Editor de Código: Visual Studio Code (VSCode), Instalación de Anaconda, Instalación de Paquetes con Conda y Pip, Conclusión

## Importancia de la terminal en Data Analytics
- **Scaling down & up:** desde pruebas locales a servidores/cluster con los mismos comandos.
- **Reproducibilidad:** un comando documentado = un paso que cualquiera puede repetir.
- **Observabilidad:** inspeccionar *logs*, archivos, procesos y redes sin capa gráfica.
- **Integración:** se conecta con `git`, `ssh`, `docker`, `python`, `conda`, `spark-submit`, etc.
- **Velocidad:** para tareas como *sampleo*, *grep*, *counts* o *splits* de archivos grandes.

## Conceptos fundamentales (muy, muy básico)
- **Ruta actual (cwd):** carpeta en la que estás trabajando ahora mismo (`pwd`).
- **Rutas relativas/absolutas:** `./data/ventas.csv` vs `/home/usuario/proy/data/ventas.csv`.
- **HOME y atajos:** `~` apunta a tu carpeta personal, `..` al directorio padre.
- **Variables de entorno:** pares `CLAVE=valor` que afectan programas (p. ej. `PATH`).
- **stdin / stdout / stderr:** entrada y salidas estándar de un proceso.
- **Códigos de salida:** `0` = OK; distintos de 0 = error/condición especial.
- **Comodines (globbing):** `*.csv`, `datos_202*.csv`.
- **Citas:** `'...'` (literal), `"..."` (expansión de variables), `\` (escapar).
- **Permisos Unix:** `rwx` para **u**suario, **g**rupo, **o**tros (`chmod`, `chown`).
- **Elevación:** `sudo` (con cuidado).

## Primeros comandos esenciales
Ejecuta estas celdas y observa la salida. Si estás en Windows con PowerShell, se muestran equivalentes.

In [ ]:

# Mostrar la carpeta de trabajo (Unix/macOS/WSL/Git Bash)
pwd


**PowerShell:** `Get-Location`

In [ ]:

# Listar archivos: largo (-l) y ocultos (-a)
ls -la


**PowerShell:** `Get-ChildItem -Force`

In [ ]:

# Crear y navegar directorios de práctica (seguros en carpeta temporal del proyecto)
mkdir -p demo_terminal/data demo_terminal/out
cd demo_terminal
pwd


> Si tu shell no soporta `-p`, crea una a una: `mkdir demo_terminal` y luego `mkdir demo_terminal/data`.

## Crear, mover, copiar y borrar archivos

In [ ]:

# Crear algunos archivos de ejemplo
echo "id,producto,precio" > data/ventas.csv
echo "1,Teclado,79.9" >> data/ventas.csv
echo "2,Mouse,39.5"   >> data/ventas.csv
echo "3,Monitor,199"  >> data/ventas.csv

# Ver su contenido
head -n 5 data/ventas.csv


In [ ]:

# Copiar y mover
cp data/ventas.csv data/ventas_backup.csv
mv data/ventas_backup.csv out/ventas_bkp.csv
ls -la out


In [ ]:

# Eliminar (¡cuidado!); -i = interactivo, -r = recursivo
rm -i out/ventas_bkp.csv


## Redirecciones y Pipes
- Redireccionar salida: `>` sobreescribe, `>>` agrega.
- Conectar comandos: `|` (**pipe**) envía la salida de un comando a la entrada del siguiente.

In [ ]:

# Contar líneas y columnas con wc y cut
echo "Analizando data/ventas.csv"
wc -l data/ventas.csv

# Extraer la columna 'precio' (3ra) separada por coma y sacar estadísticos simples
cut -d',' -f3 data/ventas.csv | tail -n +2 | sort -n | uniq -c


## Filtros y búsquedas
Usaremos `grep` (o `Select-String` en PowerShell) para localizar patrones en archivos de texto/CSV.

In [ ]:

# Buscar productos cuyo nombre contenga la letra 'o' (insensible a mayúsculas)
grep -i ",.*o.*," data/ventas.csv

# Mostrar número de línea con -n
grep -ni "monitor" data/ventas.csv || echo "No encontrado"


**PowerShell equivalente:**  
```powershell
Select-String -Path "data/ventas.csv" -Pattern "monitor" -SimpleMatch
```

## Mini‑pipelines típicos de Data Analytics (sin depender de pandas)

In [ ]:

# 1) Muestra de N filas aleatorias reproducible (si 'shuf' está disponible)
# Nota: en macOS puede instalarse con coreutils o usar 'gshuf'
if command -v shuf >/dev/null 2>&1; then
  (head -n 1 data/ventas.csv && tail -n +2 data/ventas.csv | shuf -n 2) > out/sample.csv
else
  # Fallback: tomar las primeras N filas como "muestra" determinística
  (head -n 1 data/ventas.csv && tail -n +2 data/ventas.csv | head -n 2) > out/sample.csv
fi
cat out/sample.csv


In [ ]:

# 2) Filtrar filas por precio > 50 usando 'awk' si está disponible
if command -v awk >/dev/null 2>&1; then
  (head -n 1 data/ventas.csv && tail -n +2 data/ventas.csv | awk -F',' '$3>50') > out/mayores_50.csv
  echo "Filtradas (precio>50):"; cat out/mayores_50.csv
else
  echo "awk no disponible; omitiendo este ejemplo."
fi


In [ ]:

# 3) Reemplazar separador de coma por tabulación con 'tr' y alinear en columnas con 'column'
tail -n +1 data/ventas.csv | tr ',' '\t' | column -t > out/ventas_tab.txt
head -n 5 out/ventas_tab.txt


## Procesos y recursos
- **ps**: lista procesos actuales.
- **top/htop**: vista interactiva de uso de CPU/Memoria.
- **kill**: enviar señales para terminar procesos.
- **&**: ejecutar en segundo plano; **Ctrl+C**: interrumpir.
> No se ejecuta aquí para evitar procesos colgados en el entorno del cuaderno.

## Permisos y compresión
- `chmod u+x script.sh` → hacer ejecutable.
- `tar -czf datos.tgz data/` → comprimir carpeta.
- `tar -xzf datos.tgz` → descomprimir.

## Redes rápidas
- `ping dominio.com` para conectividad básica.
- `curl` o `wget` para descargar recursos o chequear APIs (`curl -I` para encabezados).
> En clase: **no** es necesario acceder a internet; se incluye por completitud.

## Buenas prácticas
- Guarda *snippets* útiles en scripts `.sh` y documenta parámetros.
- Usa `set -euo pipefail` en scripts para detenerte en errores.
- Versiona tus scripts con `git` y agrega README con ejemplos de ejecución.
- Nombra archivos y carpetas de forma consistente y *machine-friendly* (minúsculas, guiones).
- En Windows, elige **PowerShell** o **WSL** para experiencia moderna.

## Ejercicios (progresión)
### Nivel 1 — Fundamentos
1. Crea `proyecto_cli/{data,out,docs}` y navega entre carpetas.  
2. Genera `data/productos.csv` con 5 filas inventadas (id,nombre,precio).  
3. Copia y mueve el archivo a `out/` y bórralo con confirmación.

### Nivel 2 — Pipes y filtros
4. Cuenta cuántas filas (sin encabezado) tiene tu CSV.  
5. Extrae solo la columna `precio` y obtén **mínimo y máximo** con `sort -n`.  
6. Genera un **muestreo** de 3 filas sin encabezado (usa `shuf` si lo tienes).

### Nivel 3 — Búsquedas y transformaciones
7. Filtra las filas cuyo `nombre` contenga la letra **a** o **e** (insensible a mayúsculas).  
8. Reemplaza `,` por `\t` y alinea columnas con `column -t`.  
9. Crea un pipeline que guarde a `out/filtrado.txt` lo siguiente: *encabezado + filas con precio > 50*.

### Nivel 4 — Mini‑proyecto
10. Parte de `data/ventas.csv` (o crea uno más grande) y produce:  
    - `out/top_precios.txt` con las 3 filas de mayor precio.  
    - `out/stats.txt` con conteo de filas, precio mínimo, máximo y promedio (si tienes `awk`).  
    - Un **README.md** con los comandos que ejecutaste y breves explicaciones.

> Rúbrica sugerida: claridad de comandos (30%), correcta composición de pipes (30%), organización de carpetas/salidas (20%), explicaciones en README (20%).

## Anexo: referencia rápida (Unix) y equivalentes (PowerShell)
| Tarea | Unix | PowerShell |
|---|---|---|
| Directorio actual | `pwd` | `Get-Location` |
| Listar archivos | `ls -la` | `Get-ChildItem -Force` |
| Crear carpeta | `mkdir data` | `New-Item -ItemType Directory data` |
| Copiar | `cp a b` | `Copy-Item a b` |
| Mover | `mv a b` | `Move-Item a b` |
| Borrar | `rm a` | `Remove-Item a` |
| Ver inicio de archivo | `head -n 5 a.txt` | `Get-Content a.txt -TotalCount 5` |
| Ver final de archivo | `tail -n 5 a.txt` | `Get-Content a.txt -Tail 5` |
| Buscar patrón | `grep -i "patrón" a.txt` | `Select-String -Path a.txt -Pattern "patrón"` |

## Cierre
La terminal es una **herramienta multiplicadora** para Data Analytics: permite verificar, probar y transformar datos en segundos, guionizar procesos y llevarlos desde tu laptop hasta servidores y *pipelines* productivos con cambios mínimos.

> Próximo paso sugerido: integra estos comandos con **git** y **Makefile** o scripts para automatizar *end‑to‑end*.